In [6]:
from llama_index.core import SimpleDirectoryReader
from llama_index.readers.file import PDFReader
from llama_index.core.node_parser import SentenceSplitter, HierarchicalNodeParser
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.core.schema import Node
from dotenv import load_dotenv


EMBED_MODEL =''

EMBED_DIM = 768
parent_splitter = SentenceSplitter(chunk_size=512, chunk_overlap=128, include_prev_next_rel=True)
child_splitter = SentenceSplitter(chunk_size=128, chunk_overlap=32, include_prev_next_rel=True)
node_parser = HierarchicalNodeParser(
    chunk_sizes=[512, 128],
    node_parser_ids=["parent_chunk", "child_chunk"],
    node_parser_map={"parent_chunk": parent_splitter, "child_chunk": child_splitter},
    )


In [7]:
def load_and_create_nodes_from_pdf(path: str):
    docs = PDFReader().load_data(file=path)
    nodes = node_parser.get_nodes_from_documents(docs)
    return nodes



In [8]:
nodes = load_and_create_nodes_from_pdf('/Users/amagyei/Documents/trust-manuals/C-A-Manual-2025.pdf')

In [9]:
import json
from data_normalizers.normalise_nodes import load_config, normalise_pipeline
cfg = load_config('data_normalizers/configs/c-a_manual.yaml')

clean_nodes, discard_log = normalise_pipeline(nodes, cfg, verbose=True)
print(f"Kept {len(clean_nodes)} nodes, discarded {len(discard_log)} nodes.")

[normalise] Starting with 583 nodes
[normalise] Phase 0: stripping structural header blocks...
[normalise]   Header blocks stripped from all nodes
[normalise] Phase 1: scoring remaining repeated lines...
[normalise]   Found 6 additional boilerplate lines
             └─ 'Complaints & Adjudication Departmental Manual'
             └─ 'December 9, 2025'
             └─ 'Document No. SSQM 28'
             └─ 'Issue No.         2.0'
             └─ 'Policy No. SSP/CAD-3'
             └─ 'Risk & Quality Mgt. Dept.'
[normalise] Phase 2: cleaning and filtering nodes...
[normalise]   Kept: 446  |  Discarded: 137
             └─ too_short (73 chars): 80
             └─ dot_leader_toc_fragment: 40
             └─ too_short (72 chars): 9
             └─ too_short (78 chars): 2
             └─ too_short (63 chars): 2
             └─ too_short (39 chars): 1
             └─ too_short (40 chars): 1
             └─ too_short (62 chars): 1
             └─ too_short (74 chars): 1
[normalise] Phase 3: re